# Train a decision-aligned ModernBERT router

This notebook is the **experiment driver**; reusable logic lives in `src/llm_router`. It answers one deployment question: *when can a faster model replace the strongest model without lowering observed answer quality?*

The workflow is deliberately linear: audit immutable v3 evidence, build fallback-relative targets, train ModernBERT with rank-4 LoRA, select a calibrated policy on validation, open the sealed test, and export a standalone inference artifact. A fallback-only result is valid when no safe net speedup is demonstrated.

> Run on the same GPU type recorded by v3. Router overhead and candidate latency are otherwise not comparable.

In [ ]:
# Start Jupyter/Colab from the repository root, then install the package.
%pip install -q -U -e ".[notebook,quantization]"

## 1. Reproducible experiment setup

All settings that can change training or deployment live in `RouterConfig` and are fingerprinted with the reused evidence. The GPU assertion protects the latency comparison.

In [ ]:
import os, time
from pathlib import Path

import pandas as pd
import peft, sklearn, torch, transformers
from IPython.display import display
from google.colab import drive

from llm_router import DEFAULT_CONFIG
from llm_router.models.modernbert_router import build_trainable_router
from llm_router.utils.artifacts import prepare_experiment, export_experiment
from llm_router.utils.data import evidence_summary, load_and_audit_evidence, prepare_router_data
from llm_router.utils.evaluation import evaluate_router
from llm_router.utils.training import gpu_compute_dtype, seed_everything, train_router

CONFIG = DEFAULT_CONFIG
CONFIG.validate()
seed_everything(CONFIG.seed)
assert torch.cuda.is_available(), "Choose a GPU runtime in Colab."
assert transformers.__version__ == "4.53.1"

drive.mount("/content/drive")
V3_ROOT = Path("/content/drive/MyDrive/llm_router_v3")
os.environ["HF_HOME"] = "/content/hf_cache"
GPU = torch.cuda.get_device_name(0)
DTYPE = gpu_compute_dtype()
print({"gpu": GPU, "dtype": str(DTYPE), "evidence_root": str(V3_ROOT)})

## 2. Audit evidence and construct routing targets

The audit requires 900 unique prompts and one successful measurement for each of three candidates (2,700 rows). Splitting happens at prompt level. The fallback is chosen using **training accuracy only**; alternatives are labeled safe relative to that fallback.

In [ ]:
evidence = load_and_audit_evidence(V3_ROOT, CONFIG, current_gpu=GPU)
data = prepare_router_data(evidence, CONFIG)
paths, contract, router_fingerprint = prepare_experiment(
    evidence, CONFIG, GPU, str(DTYPE),
    packages={
        "torch": torch.__version__, "transformers": transformers.__version__,
        "peft": peft.__version__, "sklearn": sklearn.__version__,
    },
)

split_counts = {
    name: data.table.loc[mask].groupby("task").size()
    for name, mask in data.masks.items()
}
display(evidence_summary(evidence).round(3))
display(pd.DataFrame(split_counts).astype(int))
print({
    "fallback": data.fallback_name,
    "alternatives": data.nonfallback_names,
    "safe_rates_train": dict(zip(
        data.nonfallback_names,
        data.replacement_safe[data.masks["train"]].mean(axis=0).round(3),
    )),
    "evidence_tag": evidence.fingerprint[:16],
    "router_tag": router_fingerprint[:16],
})

## 3. Build and train ModernBERT

ModernBERT's base weights remain frozen. Rank-4 LoRA adapters and three small heads learn replacement safety, direct latency, and token count. Every epoch is ranked by the deployed decision—not merely validation loss—after calibration, threshold search, latency blending, and measured batch-one overhead.

In [ ]:
load_started = time.perf_counter()
router, tokenizer = build_trainable_router(
    CONFIG,
    nonfallback_count=len(data.nonfallback_names),
    model_count=len(CONFIG.model_names),
)
router = router.to("cuda")
router_load_time_s = time.perf_counter() - load_started
router.encoder.print_trainable_parameters()

training = train_router(
    router, tokenizer, data, CONFIG, DTYPE, reports_dir=paths.reports
)
display(training.history.round(4))
print({"best_epoch": training.best_epoch, "training_time_s": round(training.training_time_s, 1)})

## 4. Freeze validation choices and open the sealed test

Validation fits per-candidate Platt scaling, candidate-specific safety thresholds, and the neural/task-median latency blend. Only then is test inference run. If validation cannot retain 98% quality with positive net savings, the deployed test policy is the fallback with zero router overhead.

In [ ]:
result = evaluate_router(router, tokenizer, data, CONFIG, DTYPE)
display(result.evaluation.round(4))
display(result.calibration_diagnostics.round(4))
display(result.latency_diagnostics.round(4))
display(result.confidence_intervals.round(4))
print({
    "router_active": result.router_active,
    "thresholds": dict(zip(data.nonfallback_names, result.selected_thresholds.round(4))),
    "latency_blend": result.selected_latency_blend,
    "diagnostic_overhead_ms": round(result.mean_diagnostic_overhead_ms, 3),
})

## 5. Export reports and standalone inference artifact

The artifact contains the LoRA adapter, three heads, tokenizer, calibration coefficients, thresholds, latency blend, training-only latency baselines, and the deployment guard. `ModernBERTRouterInference.from_artifact(...)` reconstructs it without notebook globals.

In [ ]:
artifact_dir = export_experiment(
    evidence=evidence, data=data, config=CONFIG, paths=paths,
    contract=contract, router_fingerprint=router_fingerprint,
    model=router, tokenizer=tokenizer, training=training, evaluation=result,
    router_load_time_s=router_load_time_s,
)
print("Reports:", paths.reports)
print("Inference artifact:", artifact_dir)

## Interpretation checklist

A positive result needs at least 98% sealed-test quality retention, positive overhead-inclusive latency reduction, and defensible non-fallback usage. If routing is disabled, inspect Brier scores (safety calibration), latency MAE/R² (latency prediction), eligible-alternative rates (selector conservatism), and oracle performance (available routing opportunity). Controlled warm batch-size-one measurements assume candidate workers are already loaded.